# Notebook 05: Human-in-the-Loop Workflows and Advanced Streaming

## Learning Objectives

1. Implement human-in-the-loop patterns with `interrupt()` + `Command(resume=...)`
2. Understand static vs dynamic interrupts
3. Build approval workflows with the modern LangGraph 1.1.9 pattern
4. Stream graph execution using all 7 streaming modes
5. Use Type-Safe Streaming v2 with `version="v2"`

## Prerequisites

- Completed Notebooks 01-04
- Understanding of checkpointing and state management

## 1. Introduction to Human-in-the-Loop

### Why Human-in-the-Loop?

AI agents need human oversight for:
- ✅ Approval before critical actions (deleting data, sending emails)
- 📝 Content review and editing
- 🔍 Verification of facts and logic
- 🎯 Course correction during execution

### LangGraph's Interrupt System

- **Static Interrupts**: `interrupt_before` / `interrupt_after` at node boundaries — set at compile time
- **Dynamic Interrupts**: `interrupt()` function inside nodes — conditional and context-aware (recommended)
- **Resumption**: Use `Command(resume=...)` — the modern canonical approach in LangGraph 1.1.9

### Flow Pattern

```
agent → process → interrupt() → [PAUSE FOR HUMAN]
                                      ↓
                              [Human provides input]
                                      ↓
                         Command(resume=value) → continue → end
```

### Key Imports

```python
from langgraph.types import interrupt, Command
```

In [ ]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
import sqlite3

load_dotenv()
print("✅ Ready for human-in-the-loop workflows with LangGraph 1.1.9!")

✅ Ready for human-in-the-loop workflows with LangGraph 1.1.9!


## The Modern HITL Pattern: interrupt() + Command(resume=...)

LangGraph 1.1.9 uses `interrupt()` inside nodes with `Command(resume=...)` for resumption. This is the canonical approach:

```python
from langgraph.types import interrupt, Command
```

**Critical rules for `interrupt()`:**
1. ⛔ **NEVER wrap in try/except** — this breaks the resumption mechanism
2. ♻️ **Code before `interrupt()` re-executes on resume** — design for idempotency (calling twice should be safe)
3. 🔑 **Requires a checkpointer** — state must be saved before the interrupt
4. 📌 **Resume values matched by position** — keep interrupt order consistent across resumptions

### Resumption Pattern

```python
# OLD approach (do not use):
graph.update_state(config, {"approved": True})
graph.invoke(None, config)

# NEW approach — Command(resume=...) is the canonical method:
final = graph.invoke(Command(resume="yes"), config)
```

## 2. Example 1: Approval Workflow with interrupt() + Command(resume=...)

In [12]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict

class ApprovalState(TypedDict):
    task: str
    plan: str
    approved: bool
    result: str

def plan_task(state: ApprovalState) -> dict:
    """Generate a plan and ask for approval."""
    plan = f"I will execute: {state['task']} using the following steps: 1. Setup, 2. Execute, 3. Verify"
    print(f"📋 Generated plan: {plan}")
    
    # Request human approval — execution pauses here
    # ⚠️ Note: plan_task() will re-run on resume — idempotency matters!
    approval = interrupt({
        "question": "Do you approve this plan?",
        "plan": plan,
        "options": ["yes", "no", "modify"],
    })
    
    print(f"👤 Human responded: {approval}")
    return {"plan": plan, "approved": approval == "yes"}

def execute_task(state: ApprovalState) -> dict:
    if not state["approved"]:
        return {"result": "❌ Task cancelled by user"}
    result = f"✅ Successfully executed: {state['task']}"
    print(result)
    return {"result": result}

# Build graph
db = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(db)

builder = StateGraph(ApprovalState)
builder.add_node("plan", plan_task)
builder.add_node("execute", execute_task)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)

app = builder.compile(checkpointer=memory)
print("✅ Approval workflow (modern pattern) ready!")

✅ Approval workflow (modern pattern) ready!


In [13]:
config = {"configurable": {"thread_id": "task-001"}}

# --- STEP 1: Run until interrupt ---
print("=== Step 1: Starting task ===")
result = app.invoke(
    {"task": "Deploy database migration", "plan": "", "approved": False, "result": ""},
    config
)

# The result contains the interrupt info
print(f"\nGraph paused. Interrupt data: {result.get('__interrupt__', 'check state')}")

# --- STEP 2: Inspect state while paused ---
state = app.get_state(config)
print(f"\nCurrent state: task='{state.values['task']}'")
print("Waiting for human approval...")

# --- STEP 3: Resume with human decision using Command(resume=...) ---
print("\n=== Step 3: Resuming with approval ===")
final_result = app.invoke(Command(resume="yes"), config)
print(f"\nFinal result: {final_result['result']}")

=== Step 1: Starting task ===
📋 Generated plan: I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify

Graph paused. Interrupt data: [Interrupt(value={'question': 'Do you approve this plan?', 'plan': 'I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify', 'options': ['yes', 'no', 'modify']}, id='e22ba66116c21563ef5301bc47d1cd76')]

Current state: task='Deploy database migration'
Waiting for human approval...

=== Step 3: Resuming with approval ===
📋 Generated plan: I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify
👤 Human responded: yes
✅ Successfully executed: Deploy database migration

Final result: ✅ Successfully executed: Deploy database migration


In [21]:
import sqlite3
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

# ── State ──────────────────────────────────────────────────────────────────
class ApprovalState(TypedDict):
    task: str
    plan: str
    approved: bool
    result: str

# ── Nodes ──────────────────────────────────────────────────────────────────
def plan_task(state: ApprovalState) -> dict:
    plan = (
        f"Plan for '{state['task']}':\n"
        "  Step 1 — Validate inputs\n"
        "  Step 2 — Execute operation\n"
        "  Step 3 — Verify results"
    )
    print(f"\n📋 Generated plan:\n{plan}\n")
    approval = interrupt({
        "question": "Do you approve this plan?",
        "options": ["yes", "no", "modify"],
    })
    print(f"👤 Human responded: {approval}")
    return {"plan": plan, "approved": (approval == "yes")}

def execute_task(state: ApprovalState) -> dict:
    if not state["approved"]:
        return {"result": "❌ Task cancelled by user."}
    return {"result": f"✅ Executed: {state['task']}"}

# ── Graph ──────────────────────────────────────────────────────────────────
db = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(db)

builder = StateGraph(ApprovalState)
builder.add_node("plan", plan_task)
builder.add_node("execute", execute_task)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)
app = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "approval-live-01"}}

# ── Run to interrupt ───────────────────────────────────────────────────────
app.invoke(
    {"task": "Deploy database migration", "plan": "", "approved": False, "result": ""},
    config,
)

# ── Prompt the human ───────────────────────────────────────────────────────
human_answer = input("⌨️  Your decision (yes / no / modify): ").strip().lower()

# ── Resume the graph with the real human input ─────────────────────────────
final = app.invoke(Command(resume=human_answer), config)
print(f"\n🏁 Final result: {final['result']}")


📋 Generated plan:
Plan for 'Deploy database migration':
  Step 1 — Validate inputs
  Step 2 — Execute operation
  Step 3 — Verify results


📋 Generated plan:
Plan for 'Deploy database migration':
  Step 1 — Validate inputs
  Step 2 — Execute operation
  Step 3 — Verify results

👤 Human responded: yes

🏁 Final result: ✅ Executed: Deploy database migration


## Static vs Dynamic Interrupts

LangGraph supports two interrupt styles:

### Static Interrupts (compile-time)
Set breakpoints when compiling the graph — always interrupt at these nodes:

```python
app = graph.compile(
    checkpointer=memory,
    interrupt_before=["execute_payment"],   # Pause BEFORE this node runs
    interrupt_after=["classify_intent"],    # Pause AFTER this node runs
)
```
- Good for: Fixed review gates, testing/debugging specific nodes
- Limitations: Can't be conditional, always triggers

### Dynamic Interrupts (runtime, recommended)
The `interrupt()` function inside nodes — conditional and context-aware:

```python
def execute_payment(state):
    if state["amount"] > 1000:
        approval = interrupt(f"Large payment of ${state['amount']} — approve?")
        if approval != "yes":
            return {"status": "declined"}
    # Proceed with payment...
```
- Good for: Business logic-driven pauses, conditional approval
- More flexible: Can carry context, conditional on state

### Resumption Differences

| Style | How to resume |
|---|---|
| Static (`interrupt_before/after`) | `app.invoke(None, config)` — pass `None` to continue |
| Dynamic (`interrupt()`) | `app.invoke(Command(resume=value), config)` — pass the resume value |

## 3. Static Interrupts: interrupt_before Example

In [14]:
# Demonstrating static interrupt_before
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import TypedDict

class PaymentState(TypedDict):
    amount: float
    merchant: str
    status: str

def validate_payment(state: PaymentState) -> dict:
    print(f"💳 Validating payment: ${state['amount']} to {state['merchant']}")
    return {"status": "validated"}

def process_payment(state: PaymentState) -> dict:
    print(f"✅ Processing payment: ${state['amount']}")
    return {"status": "processed"}

db2 = sqlite3.connect(":memory:", check_same_thread=False)
memory2 = SqliteSaver(db2)

builder2 = StateGraph(PaymentState)
builder2.add_node("validate", validate_payment)
builder2.add_node("process", process_payment)
builder2.add_edge(START, "validate")
builder2.add_edge("validate", "process")
builder2.add_edge("process", END)

# Static interrupt: ALWAYS pause before "process"
app2 = builder2.compile(
    checkpointer=memory2,
    interrupt_before=["process"]  # ← static breakpoint
)

config2 = {"configurable": {"thread_id": "payment-001"}}

# Run to the interrupt
print("=== Running to interrupt ===")
app2.invoke({"amount": 5000.0, "merchant": "AWS", "status": "pending"}, config2)

print("\n⏸️  Paused before 'process' node. Human review required.")

# For static interrupts, resume by invoking with None (no Command needed)
print("\n=== Resuming ===")
final = app2.invoke(None, config2)
print(f"Final status: {final['status']}")

=== Running to interrupt ===
💳 Validating payment: $5000.0 to AWS

⏸️  Paused before 'process' node. Human review required.

=== Resuming ===
✅ Processing payment: $5000.0
Final status: processed


## 4. Streaming Modes — Complete Reference

LangGraph 1.1.9 provides 7 streaming modes. You can also combine them.

| Mode | What you receive | Best for |
|---|---|---|
| `"values"` | Full state after every node completes | Displaying current state |
| `"updates"` | Only the keys that changed in each node | Efficient updates, diffs |
| `"messages"` | LLM tokens as they stream + metadata | Chat interfaces, token streaming |
| `"custom"` | Data you explicitly send with `get_stream_writer()` | Custom progress events |
| `"debug"` | Checkpoint + task lifecycle events with metadata | Debugging |
| `"checkpoints"` | State snapshots (requires checkpointer) | Audit logs, state inspection |
| `"tasks"` | Node start/finish events | Progress tracking |

You can combine multiple modes by passing a list: `stream_mode=["updates", "custom"]`

In [15]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

def step1(state: MessagesState) -> dict:
    return {"messages": [{"role": "assistant", "content": "Step 1 complete"}]}

def step2(state: MessagesState) -> dict:
    return {"messages": [{"role": "assistant", "content": "Step 2 complete"}]}

stream_builder = StateGraph(MessagesState)
stream_builder.add_node("step1", step1)
stream_builder.add_node("step2", step2)
stream_builder.add_edge(START, "step1")
stream_builder.add_edge("step1", "step2")
stream_builder.add_edge("step2", END)
stream_app = stream_builder.compile()

print("=== stream_mode='values' — full state after each node ===")
for chunk in stream_app.stream(
    {"messages": [HumanMessage("Go")]},
    stream_mode="values"
):
    print(f"State has {len(chunk['messages'])} messages")

print("\n=== stream_mode='updates' — only changes from each node ===")
for chunk in stream_app.stream(
    {"messages": [HumanMessage("Go")]},
    stream_mode="updates"
):
    print(f"Node update: {chunk}")

=== stream_mode='values' — full state after each node ===
State has 1 messages
State has 2 messages
State has 3 messages

=== stream_mode='updates' — only changes from each node ===
Node update: {'step1': {'messages': [{'role': 'assistant', 'content': 'Step 1 complete'}]}}
Node update: {'step2': {'messages': [{'role': 'assistant', 'content': 'Step 2 complete'}]}}


In [16]:
# Custom streaming with get_stream_writer()
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

class ProcessState(TypedDict):
    items: list
    processed: list

def process_items(state: ProcessState) -> dict:
    """Node that emits custom progress events during processing."""
    writer = get_stream_writer()  # Get the stream writer

    processed = []
    for i, item in enumerate(state["items"]):
        result = f"processed_{item}"
        processed.append(result)

        # Emit custom progress event
        writer({"type": "progress", "item": item, "done": i + 1, "total": len(state["items"])})

    return {"processed": processed}

custom_builder = StateGraph(ProcessState)
custom_builder.add_node("process", process_items)
custom_builder.add_edge(START, "process")
custom_builder.add_edge("process", END)
custom_app = custom_builder.compile()

print("=== stream_mode='custom' — custom events from nodes ===")
for event in custom_app.stream(
    {"items": ["alpha", "beta", "gamma"], "processed": []},
    stream_mode="custom"
):
    print(f"Progress: {event['item']} ({event['done']}/{event['total']})")

=== stream_mode='custom' — custom events from nodes ===
Progress: alpha (1/3)
Progress: beta (2/3)
Progress: gamma (3/3)


In [17]:
# Combining multiple stream modes
print("=== Combining stream modes: ['updates', 'custom'] ===")
for chunk in custom_app.stream(
    {"items": ["x", "y", "z"], "processed": []},
    stream_mode=["updates", "custom"]  # ← list of modes
):
    # When combining modes, each chunk is a tuple: (mode_name, data)
    print(f"Chunk: {chunk}")

=== Combining stream modes: ['updates', 'custom'] ===
Chunk: ('custom', {'type': 'progress', 'item': 'x', 'done': 1, 'total': 3})
Chunk: ('custom', {'type': 'progress', 'item': 'y', 'done': 2, 'total': 3})
Chunk: ('custom', {'type': 'progress', 'item': 'z', 'done': 3, 'total': 3})
Chunk: ('updates', {'process': {'processed': ['processed_x', 'processed_y', 'processed_z']}})


In [18]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)

def chat(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

chat_builder = StateGraph(MessagesState)
chat_builder.add_node("chat", chat)
chat_builder.add_edge(START, "chat")
chat_builder.add_edge("chat", END)
chat_app = chat_builder.compile()

print("=== stream_mode='messages' — LLM tokens as they arrive ===")
print("Response: ", end="", flush=True)
for chunk, metadata in chat_app.stream(
    {"messages": [HumanMessage("Count from 1 to 5, one number per word.")]},
    stream_mode="messages"
):
    if hasattr(chunk, "content") and chunk.content:
        print(chunk.content, end="", flush=True)
print()  # newline after streaming

=== stream_mode='messages' — LLM tokens as they arrive ===
Response: One  
Two  
Three  
Four  
Five  


## 5. Mini Project: Research Assistant with HITL + Streaming

Build an assistant that:
- Searches for information
- Pauses for human clarification using `interrupt()` + `Command(resume=...)`
- Gets approval before expensive operations
- Streams progress updates

In [19]:
from langchain_core.tools import tool

class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    findings: list
    approved_sources: list

@tool
def search_papers(query: str) -> str:
    """Search for academic papers (simulated)."""
    papers = [
        "Paper 1: Introduction to Quantum Computing",
        "Paper 2: Quantum Algorithms Overview",
        "Paper 3: Practical Quantum Applications"
    ]
    return "\n".join(papers)

@tool
def summarize_paper(title: str) -> str:
    """Summarize a paper (simulated, expensive operation)."""
    return f"Summary of {title}: [Detailed summary would go here]"

def research_node(state: ResearchState) -> dict:
    """Conduct research with human oversight."""
    # Search for papers
    results = search_papers.invoke({"query": state["topic"]})
    
    # Ask for clarification if needed
    clarification = interrupt(
        f"Found these papers about {state['topic']}. \n{results}\n\n"
        "Should I continue with detailed analysis? (yes/no)"
    )
    
    if clarification == "yes":
        # Proceed with analysis
        findings = ["Finding 1", "Finding 2", "Finding 3"]
        return {"findings": findings}
    
    return {"findings": []}

# Build research assistant
research_builder = StateGraph(ResearchState)
research_builder.add_node("research", research_node)
research_builder.add_edge(START, "research")
research_builder.add_edge("research", END)

research_db = sqlite3.connect(":memory:", check_same_thread=False)
research_memory = SqliteSaver(research_db)
research_assistant = research_builder.compile(checkpointer=research_memory)

print("✅ Research Assistant ready!")

✅ Research Assistant ready!


## 6. Type-Safe Streaming v2 (LangGraph 1.1.0+)

LangGraph 1.1.0 introduced type-safe streaming with `version="v2"`. Each chunk becomes a self-describing `StreamPart` TypedDict, making it easier to handle multiple stream types:

- Each chunk self-identifies its type — no guessing what kind of chunk you received
- `invoke()` returns `GraphOutput` with `.value` (typed state) and `.interrupts` (interrupt objects)
- Subgraph chunks include `ns` (namespace) field — know which subgraph emitted the event
- Fully backward compatible — default is still `version="v1"`

In [20]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm_v2 = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def chat_v2(state: MessagesState) -> dict:
    response = llm_v2.invoke(state["messages"])
    return {"messages": [response]}

v2_builder = StateGraph(MessagesState)
v2_builder.add_node("chat", chat_v2)
v2_builder.add_edge(START, "chat")
v2_builder.add_edge("chat", END)
v2_app = v2_builder.compile()

print("=== version='v2' streaming — typed StreamPart chunks ===")
for chunk in v2_app.stream(
    {"messages": [HumanMessage("Say hello")]},
    stream_mode="updates",
    version="v2"  # ← opt-in to type-safe streaming
):
    # chunk is now a StreamPart TypedDict:
    # {"type": "updates"|"values"|..., "ns": (), "data": ...}
    print(f"Type: {chunk['type']}, Namespace: {chunk['ns']}, Data keys: {list(chunk['data'].keys())}")

print("\n=== version='v2' invoke — returns GraphOutput dataclass ===")
result = v2_app.invoke(
    {"messages": [HumanMessage("Say hello")]},
    version="v2"
)
# result is now a GraphOutput, not a plain dict:
print(f"Type: {type(result)}")
print(f"State value: {result.value['messages'][-1].content}")
print(f"Interrupts: {result.interrupts}")  # Any interrupt objects if graph was paused

=== version='v2' streaming — typed StreamPart chunks ===
Type: updates, Namespace: (), Data keys: ['chat']

=== version='v2' invoke — returns GraphOutput dataclass ===
Type: <class 'langgraph.types.GraphOutput'>
State value: Hello! How can I assist you today?
Interrupts: ()


## 7. Key Takeaways

### Concepts Mastered

1. **interrupt() + Command(resume=...)**: The modern canonical pattern for pausing and resuming graphs in LangGraph 1.1.9
2. **Static vs Dynamic Interrupts**: Static at compile-time (`interrupt_before/after`), dynamic at runtime (`interrupt()` inside nodes)
3. **7 Streaming Modes**: `values`, `updates`, `messages`, `custom`, `debug`, `checkpoints`, `tasks` — combinable
4. **Type-Safe Streaming v2**: `version="v2"` gives typed `StreamPart` chunks and `GraphOutput` from `invoke()`
5. **Approval Workflows**: Human oversight for critical operations using context-rich interrupt payloads

### Best Practices

✅ **Use `Command(resume=...)` for resumption** — not `update_state()` (deprecated pattern)  
✅ **Never wrap `interrupt()` in try/except** — it breaks the resumption mechanism  
✅ **Design nodes for idempotency** — code before `interrupt()` re-executes on resume  
✅ **Provide context in interrupt messages** — help humans make informed decisions  
✅ **Stream for long-running operations** — keep users informed with appropriate mode  
✅ **Always use a checkpointer** — required for both interrupts and streaming checkpoints  

### Streaming Mode Quick Reference

| Use case | Recommended mode |
|---|---|
| Show current state after each step | `"values"` |
| Show only what changed | `"updates"` |
| Real-time LLM token output | `"messages"` |
| Custom progress events | `"custom"` |
| Debugging graph execution | `"debug"` |
| Multiple at once | `stream_mode=["updates", "custom"]` |
| Type-safe chunks with metadata | `version="v2"` |

### What's Next?

In **Notebook 06**, you'll learn about parallel execution, subgraphs, and multi-agent systems!